# Live Training Loop Diagnostics

## Install

```bash
pip install graphyco
```

---

## Setup & Imports

Import live training monitor and diagnostics tools.

In [ ]:
import sys, os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from graphyco.bridge.grad_bridge import LiveTrainingMonitor
from graphyco.visualizer import LiveTrainingDiagnostics, GradientHealthStatus

torch.manual_seed(42)

## LiveTrainingMonitor in Active Training Loop

Observe training steps with zero distortion of forward or backward execution.

In [ ]:
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Attach monitor observing every 5 steps
monitor = LiveTrainingMonitor(model, log_interval=5)

for step in range(25):
    x = torch.randn(16, 64)
    y = torch.randint(0, 10, (16,))
    
    with monitor.observe(step):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    
    if monitor.has_new_data():
        bneck = monitor.get_latest_bottleneck()
        print(f"Step {step:2d} | Loss: {loss.item():.4f} | Peak Choke Node: {bneck['bottleneck_node']} (G_max={bneck['max_concentration']:.2f})")

## Real-Time Health Diagnostics with LiveTrainingDiagnostics

Track per-layer gradients, activations, dead neurons, and health alerts.

In [ ]:
diag_model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)
diag_opt = optim.SGD(diag_model.parameters(), lr=0.01)

diag = LiveTrainingDiagnostics(diag_model, vanishing_threshold=1e-6, exploding_threshold=50.0)

for step in range(15):
    x = torch.randn(8, 32)
    y = torch.randint(0, 2, (8,))
    with diag.observe_step(step):
        diag.current_loss = 0.5 / (step + 1)
        diag_opt.zero_grad()
        loss = criterion(diag_model(x), y)
        loss.backward()
        diag_opt.step()

snapshot = diag.get_snapshot()
print(f"Health Status      : {snapshot['health_status']}")
print(f"Active Bottleneck  : {snapshot['active_bottleneck_layer']} (G_max={snapshot['gmax']:.2f})")
print(f"Average Step Time  : {snapshot['avg_step_time_ms']:.2f} ms")

## Layer Telemetry Inspection

Inspect gradient RMS, activation RMS, and dead neuron percentages across submodules.

In [ ]:
layer_rows = []
for layer_name, stats in snapshot["layers"].items():
    layer_rows.append({
        "Layer": layer_name,
        "Latest Grad RMS": f"{stats['latest_grad_rms']:.6f}",
        "Latest Act RMS": f"{stats['latest_act_rms']:.6f}",
        "Dead Neuron Ratio": f"{stats['dead_neuron_ratio'] * 100:.1f}%"
    })
pd.DataFrame(layer_rows)

## Automated Anomaly Detection Trigger

Inject vanishing gradient signal and verify automated anomaly logging.

In [ ]:
anomaly_model = nn.Sequential(nn.Linear(16, 16), nn.Linear(16, 2))
anomaly_diag = LiveTrainingDiagnostics(anomaly_model, vanishing_threshold=1e-4)

x = torch.randn(4, 16)
with anomaly_diag.observe_step(1):
    anomaly_diag.current_loss = 0.01
    out = anomaly_model(x) * 1e-6
    loss = out.sum()
    loss.backward()

snap_anomaly = anomaly_diag.get_snapshot()
print(f"Status: {snap_anomaly['health_status']}")
print(f"Alerts: {snap_anomaly['alerts']}")